In [1]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

# 1. Connect to your SQLite database
conn = sqlite3.connect("../database/superstore.db")

# 2. Write and execute a SQL query using CTEs / Window functions
query = """
    SELECT 
        sales_month,
        category,
        total_sales,
        total_profit,
        ROUND(AVG(total_sales) OVER (
            PARTITION BY category 
            ORDER BY sales_month 
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 2) AS rolling_3m_avg_sales
    FROM monthly_sales_summary
    ORDER BY sales_month ASC, category;
"""

df = pd.read_sql_query(query, conn)
conn.close()

# 3. View the queried dataset
print(df.head(10))


  sales_month         category  total_sales  total_profit  \
0     2009-01        Furniture     44641.24       7090.96   
1     2009-01        Furniture     26133.39     -11053.60   
2     2009-01        Furniture     56684.24       -244.64   
3     2009-01        Furniture     22486.15       2045.06   
4     2009-01        Furniture     17193.36       3587.32   
5     2009-01        Furniture     41358.17       2808.84   
6     2009-01        Furniture        11.08         -8.93   
7     2009-01  Office Supplies     21023.16       8509.98   
8     2009-01  Office Supplies      1549.08       -513.62   
9     2009-01  Office Supplies       513.13         57.88   

   rolling_3m_avg_sales  
0              44641.24  
1              35387.32  
2              42486.29  
3              35101.26  
4              32121.25  
5              27012.56  
6              19520.87  
7              21023.16  
8              11286.12  
9               7695.12  


In [2]:
import sqlite3
import pandas as pd
import numpy as np

# 1. Fetch monthly aggregated sales from SQLite
conn = sqlite3.connect("../database/superstore.db")

query = """
    SELECT 
        sales_month,
        SUM(total_sales) AS total_sales
    FROM monthly_sales_summary
    GROUP BY sales_month
    ORDER BY sales_month ASC;
"""

df_sales = pd.read_sql_query(query, conn)

# 2. Simple Moving Average & Exponential Growth Forecast
df_sales['sales_3m_moving_avg'] = df_sales['total_sales'].rolling(window=3).mean()

# Calculate simple month-over-month growth rate for future projections
df_sales['growth_rate'] = df_sales['total_sales'].pct_change()
avg_growth = df_sales['growth_rate'].mean()

# Generate 3 future forecast months
last_month = pd.to_datetime(df_sales['sales_month'].iloc[-1])
future_dates = [ (last_month + pd.DateOffset(months=i)).strftime('%Y-%m') for i in range(1, 4) ]

last_sales = df_sales['total_sales'].iloc[-1]
future_forecasts = []

for i, date in enumerate(future_dates, 1):
    predicted_sales = round(last_sales * ((1 + avg_growth) ** i), 2)
    future_forecasts.append({'sales_month': date, 'forecast_sales': predicted_sales, 'type': 'Forecast'})

df_future = pd.DataFrame(future_forecasts)

# Combine historical actuals and forecasts
df_actuals = df_sales[['sales_month', 'total_sales']].copy()
df_actuals.columns = ['sales_month', 'forecast_sales']
df_actuals['type'] = 'Actual'

df_final_forecast = pd.concat([df_actuals, df_future], ignore_index=True)

# 3. Write predictions directly into a new SQLite table
df_final_forecast.to_sql("sales_forecasts", conn, if_exists="replace", index=False)

print("Forecast results saved to table 'sales_forecasts' in superstore.db successfully!")

# Verify by querying the new table
df_check = pd.read_sql_query("SELECT * FROM sales_forecasts ORDER BY sales_month DESC LIMIT 5;", conn)
conn.close()

print("\nLast 5 Records (Including Future Predictions):")
print(df_check)

Forecast results saved to table 'sales_forecasts' in superstore.db successfully!

Last 5 Records (Including Future Predictions):
  sales_month  forecast_sales      type
0     2013-03       383923.61  Forecast
1     2013-02       373927.52  Forecast
2     2013-01       364191.68  Forecast
3     2012-12       354709.34    Actual
4     2012-11       256020.10    Actual
